# ⛏️ Tonopah-Region Mine Explorer

Explore **every USGS-recorded mine, adit, shaft, and prospect** in Nye County, NV and all its neighbors — right from your browser.

**How to use this:** click **Runtime ▸ Run all** (Colab) or the ⏩ button (Jupyter). Wait ~1 minute for data to download. Then scroll to the **🎛️ Playground** section and change the settings to ask your own questions.

*This is your own private copy — you cannot break the original. Experiment freely. If anything goes wrong, just reload the page and Run all again.*


In [ ]:
# @title 1️⃣ Setup (installs — takes ~30s, ignore the wall of text)
%pip -q install pandas duckdb folium requests
print('✅ ready')


In [ ]:
# @title 2️⃣ Download USGS data (cached — only downloads the first time)
import os, io, zipfile, requests, pandas as pd

os.makedirs('data', exist_ok=True)

def fetch(url, name):
    path = f'data/{name}'
    if not os.path.exists(path):
        print(f'downloading {name} ...')
        r = requests.get(url, timeout=300); r.raise_for_status()
        open(path, 'wb').write(r.content)
    return path

# MRDS: worldwide mineral-site database (26 MB zip)
z = zipfile.ZipFile(fetch('https://mrdata.usgs.gov/mrds/mrds-csv.zip', 'mrds-csv.zip'))
if not os.path.exists('data/mrds.csv'):
    z.extract('mrds.csv', 'data')

# USMIN: mine features digitized from topo maps (points only, via dbf+shp)
for st in ['NV', 'CA']:
    zipfile.ZipFile(fetch(f'https://mrdata.usgs.gov/usmin/state/usmin-{st}.zip', f'usmin-{st}.zip')).extractall(f'data/usmin-{st}')
print('✅ data on disk:', sorted(os.listdir('data')))


In [ ]:
# @title 3️⃣ Load + filter to the Tonopah region
import struct, math

NV_COUNTIES = ['Nye','Esmeralda','Mineral','Churchill','Lander','Eureka','White Pine','Lincoln','Clark']
CA_COUNTIES = ['Inyo']   # add 'Mono' if you like

# --- MRDS sites (rich attributes: commodities, status, geology) ---
mrds_all = pd.read_csv('data/mrds.csv', low_memory=False)
mrds = mrds_all[((mrds_all.state=='Nevada') & (mrds_all.county.isin(NV_COUNTIES))) |
                ((mrds_all.state=='California') & (mrds_all.county.isin(CA_COUNTIES)))].copy()

# --- USMIN features: read shapefiles with a tiny pure-python reader (no GDAL needed) ---
def read_points_shp(base):
    # .shp for coords (point type only), .dbf for attributes
    with open(base + '.shp','rb') as f:
        buf = f.read()
    pts, off = [], 100
    while off < len(buf):
        _, clen = struct.unpack('>ii', buf[off:off+8])
        shp_type = struct.unpack('<i', buf[off+8:off+12])[0]
        if shp_type == 1:
            x, y = struct.unpack('<dd', buf[off+12:off+28])
            pts.append((x, y))
        else:
            pts.append((None, None))
        off += 8 + clen*2
    with open(base + '.dbf','rb') as f:
        d = f.read()
    n_rec, hdr, rlen = struct.unpack('<I', d[4:8])[0], struct.unpack('<H', d[8:10])[0], struct.unpack('<H', d[10:12])[0]
    fields, p = [], 32
    while d[p] != 0x0d:
        name = d[p:p+11].split(b'\x00')[0].decode()
        fields.append((name, d[p+16]))
        p += 32
    rows = []
    for i in range(n_rec):
        rec, q, row = d[hdr+i*rlen:hdr+(i+1)*rlen], 1, {}
        for name, ln in fields:
            row[name] = rec[q:q+ln].decode('latin-1').strip(); q += ln
        rows.append(row)
    out = pd.DataFrame(rows)
    out['lon'] = [p[0] for p in pts][:len(out)]
    out['lat'] = [p[1] for p in pts][:len(out)]
    return out

usmin = pd.concat([
    read_points_shp('data/usmin-NV/NV-point').query('COUNTY in @NV_COUNTIES'),
    read_points_shp('data/usmin-CA/CA-point').query('COUNTY in @CA_COUNTIES'),
], ignore_index=True)

# distance from Tonopah for every feature
def miles_from_tonopah(lat, lon, LAT0=38.0672, LON0=-117.2301):
    R=3958.76; p1,p2=math.radians(LAT0),math.radians(lat)
    dp,dl=math.radians(lat-LAT0),math.radians(lon-LON0)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))
usmin['miles_from_tonopah'] = [round(miles_from_tonopah(la,lo),1) if la else None for la,lo in zip(usmin.lat, usmin.lon)]
mrds['miles_from_tonopah'] = [round(miles_from_tonopah(la,lo),1) if pd.notna(la) else None for la,lo in zip(mrds.latitude, mrds.longitude)]

print(f'✅ {len(mrds):,} MRDS sites | {len(usmin):,} USMIN mapped features')


## 👀 Meet the data

Two datasets, two personalities:
- **USMIN** = *where the holes in the ground are* (adits, shafts, pits — traced off old topo maps)
- **MRDS** = *what was mined and how it panned out* (commodities, producer vs. prospect, geology)


In [ ]:
print('USMIN — physical features by type:')
display(usmin.FTR_TYPE.value_counts().head(12).to_frame('count'))
print('MRDS — sites by development status:')
display(mrds.dev_stat.value_counts().to_frame('count'))
print('MRDS — top primary commodities in the region:')
display(mrds.commod1.value_counts().head(15).to_frame('count'))


---
# 🎛️ Playground

**This is your control panel.** Change any value below, then re-run this cell and the two after it (Shift+Enter runs a cell).


In [ ]:
# ============== CHANGE ME ==============
FEATURE_TYPES = ['Adit', 'Mine Shaft']   # try: 'Prospect Pit', 'Open Pit Mine', 'Quarry', 'Tunnel'
MAX_MILES     = 50                        # radius from Tonopah (blank = no limit: None)
COUNTIES      = None                      # None = all, or e.g. ['Nye', 'Esmeralda']
NAMED_ONLY    = False                     # True = only features with a name
# =======================================

q = usmin[usmin.FTR_TYPE.isin(FEATURE_TYPES)]
if MAX_MILES:  q = q[q.miles_from_tonopah <= MAX_MILES]
if COUNTIES:   q = q[q.COUNTY.isin(COUNTIES)]
if NAMED_ONLY: q = q[q.FTR_NAME != '']
q = q.sort_values('miles_from_tonopah')

print(f'🔎 {len(q):,} features match')
display(q[['FTR_TYPE','FTR_NAME','COUNTY','STATE','miles_from_tonopah','TOPO_NAME']].head(25))


In [ ]:
# @title 🗺️ Map the matches (auto-capped at 2,000 markers)
import folium
from folium.plugins import MarkerCluster

m = folium.Map(location=[38.0672,-117.2301], zoom_start=8, tiles='OpenStreetMap')
folium.Marker([38.0672,-117.2301], tooltip='Tonopah', icon=folium.Icon(color='red', icon='star')).add_to(m)
cluster = MarkerCluster().add_to(m)
for _, r in q.head(2000).iterrows():
    folium.CircleMarker([r.lat, r.lon], radius=4, fill=True,
        popup=f"{r.FTR_NAME or '(unnamed)'}<br>{r.FTR_TYPE} — {r.COUNTY} Co.<br>{r.miles_from_tonopah} mi from Tonopah"
    ).add_to(cluster)
m


## 💰 Ask MRDS: what was mined around here?


In [ ]:
# ============== CHANGE ME ==============
COMMODITY  = 'Gold'      # try: 'Silver', 'Copper', 'Turquoise', 'Uranium', 'Lithium', 'Mercury'
STATUS     = 'Producer'  # try: 'Past Producer', 'Prospect', 'Occurrence', or None for all
MILES      = 60
# =======================================

hits = mrds[(mrds[['commod1','commod2','commod3']].apply(lambda c: c.str.contains(COMMODITY, case=False, na=False)).any(axis=1))]
if STATUS: hits = hits[hits.dev_stat.str.contains(STATUS, na=False)]
if MILES:  hits = hits[hits.miles_from_tonopah <= MILES]
hits = hits.sort_values('miles_from_tonopah')
print(f'🔎 {len(hits):,} MRDS sites match')
display(hits[['site_name','commod1','commod2','dev_stat','work_type','county','miles_from_tonopah','url']].head(25))


## 🧪 SQL corner (for the adventurous)

The same dataframes are queryable with plain SQL via DuckDB — edit away:


In [ ]:
import duckdb
duckdb.sql('''
  SELECT county, count(*) AS sites,
         sum(CASE WHEN dev_stat LIKE '%Producer%' THEN 1 ELSE 0 END) AS producers
  FROM mrds
  GROUP BY county ORDER BY sites DESC
''').df()


---
### ⚠️ A word from the safety department
These locations come from historical maps and records. Many are on private claims, collapsed, flooded, or full of bad air. Great for maps and history — **never enter an abandoned mine.**

*Data: USGS MRDS & USMIN (public domain) — mrdata.usgs.gov*
